In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [2]:
df = pd.read_parquet("/Users/jasonc/Desktop/DSC_291/merged_features.parquet")

In [3]:
df

,evaluation_date,FPF_TARGET,total_balance,agg_total_amount,agg_mean_amount,agg_std_amount,agg_min_amount,agg_max_amount,agg_median_amount,agg_transaction_count,...,fft_low_freq_power_cat35.0,fft_high_freq_power_cat35.0,fft_power_ratio_cat35.0,fft_dominant_freq_cat31.0,fft_dominant_power_cat31.0,fft_spectral_entropy_cat31.0,fft_low_freq_power_cat31.0,fft_high_freq_power_cat31.0,fft_power_ratio_cat31.0,dataset_type
masked_consumer_id,,,,,,,,,,,,,,,,,,,,,
C01100001,2021-08-26,0.0,1380.84,-8111.420000,-3.051701,645.540740,-8000.000000,8483.000000,-25.380000,2658.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,real
C01100002,2022-08-02,0.0,20163.90,-9490.730000,-3.813070,910.144154,-19150.000000,20000.000000,-25.290000,2489.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,real
C01100003,2021-03-04,0.0,3986.25,2137.770000,3.224389,748.056684,-4310.280000,4409.400000,-21.350000,663.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,real
C01100004,2022-11-19,0.0,5956.03,-29837.870000,-27.525710,414.375001,-1800.000000,1748.000000,-27.555000,1084.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,real
C01100005,2021-11-21,0.0,29421.10,-9756.370000,-4.044930,1314.772339,-6987.070000,39915.160000,-25.080000,2412.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,real
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C01104978_simulation_16,2021-10-07,1.0,0.00,-5392.700396,-5.054077,772.277000,-5024.826805,19985.779850,-34.227799,1067.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,simulation
C01104978_simulation_17,2021-10-07,1.0,0.00,-3528.719044,-3.307141,774.546109,-4966.893421,20092.994060,-41.945486,1067.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,simulation
C01104978_simulation_18,2021-10-07,1.0,0.00,-4155.106431,-3.894195,771.804247,-4984.093947,19916.403258,-38.925977,1067.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,simulation


In [4]:
# Group by customer_category and FPF_TARGET, then count occurrences
category_target_counts = (
    df.groupby(["dataset_type", "FPF_TARGET"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: "count_0", 1.0: "count_1"})
)
category_target_counts

FPF_TARGET,count_0,count_1
dataset_type,,
real,3942,58
simulation,0,1160


In [5]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score

# If masked_consumer_id is the index, make it a column:
df = df.reset_index().rename(columns={'index':'masked_consumer_id'})
# Derive customer_category from the ID prefix (assumes IDs start like 'C01_…')
df['customer_category'] = df['masked_consumer_id'].str.extract(r'^(C\d{2})')

# 1) Define customer categories and model pipelines
categories = ["C01"]

model_pipelines = {
    "XGBoost": Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.01,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=1,
            eval_metric='logloss',
            random_state=42
        ))
    ])
}

results = {}

for cat in categories:
    print(f"\n=== Category: {cat} ===")
    df_cat = df[df["customer_category"] == cat].copy()
    if df_cat.shape[0] < 10:
        print("  skipping: too few samples")
        continue

    # 2) split real vs simulation
    mask_sim = df_cat['masked_consumer_id'].str.contains(r'_simulation_')
    df_real  = df_cat.loc[~mask_sim].reset_index(drop=True)
    df_sim   = df_cat.loc[ mask_sim].reset_index(drop=True)

    # 3) real-only stratified split
    Xr = df_real.drop(columns=[
        'masked_consumer_id',
        'customer_category',
        'FPF_TARGET',
        'evaluation_date',
        'dataset_type'
    ])
    yr = df_real['FPF_TARGET']
    feat_names = Xr.columns.tolist()

    Xr_tr, Xr_val, yr_tr, yr_val = train_test_split(
        Xr, yr,
        test_size=0.2,
        stratify=yr,
        random_state=42
    )

    # 4) bring in only those sims whose parent is in the training real IDs
    df_sim['parent_id'] = df_sim['masked_consumer_id'].str.replace(r'_simulation_\d+$','', regex=True)
    train_ids = set(df_real.loc[Xr_tr.index, 'masked_consumer_id'])
    df_sim_tr = df_sim.loc[df_sim['parent_id'].isin(train_ids)].drop(columns=['parent_id'])
    X_sim = df_sim_tr[feat_names]
    y_sim = df_sim_tr['FPF_TARGET']

    # 5) assemble final train/val sets
    X_train = pd.concat([Xr_tr, X_sim], axis=0)
    y_train = pd.concat([yr_tr, y_sim], axis=0)
    X_val, y_val = Xr_val, yr_val

    # 6) fit L1 logistic to select top 50 features
    l1 = LogisticRegression(penalty='l2',
                            max_iter=2000, random_state=42)
    l1.fit(X_train, y_train)
    coefs = np.abs(l1.coef_).ravel()
    top_idx = np.argsort(coefs)[::-1][:250]
    selected_feats = [feat_names[i] for i in top_idx]
    print(f"Selected top 250 features for {cat}: {selected_feats}")

    # 7) subset to top 250
    X_train_sel = X_train.iloc[:, top_idx]
    X_val_sel   = X_val.iloc[:,   top_idx]

    # 8) train and evaluate each pipeline
    for name, pipe in model_pipelines.items():
        print(f"→ Training {name} on top 250 features")
        pipe.fit(X_train_sel, y_train)
        p_tr = pipe.predict_proba(X_train_sel)[:,1]
        p_val = pipe.predict_proba(X_val_sel)[:,1]
        train_auc = roc_auc_score(y_train, p_tr)
        val_auc   = roc_auc_score(y_val,   p_val)
        print(f"   Train AUC={train_auc:.3f}, Val AUC={val_auc:.3f}")

        # Optional: store results
        results.setdefault(cat, {})[name] = {'train_auc': train_auc, 'val_auc': val_auc}


=== Category: C01 ===


/Users/jasonc/opt/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Selected top 250 features for C01: ['fft_power_ratio_cat31.0', 'week_roll_rolling_mean_cat_mean_33.0', 'week_roll_rolling_mean_cat_mean_21.0', 'week_roll_rolling_mean_cat_mean_22.0', 'week_roll_rolling_mean_cat_mean_23.0', 'week_roll_rolling_mean_cat_mean_24.0', 'week_roll_rolling_mean_cat_mean_25.0', 'week_roll_rolling_mean_cat_mean_26.0', 'week_roll_rolling_mean_cat_mean_27.0', 'week_roll_rolling_mean_cat_mean_28.0', 'week_roll_rolling_mean_cat_mean_29.0', 'week_roll_rolling_mean_cat_mean_30.0', 'week_roll_rolling_mean_cat_mean_32.0', 'week_roll_rolling_mean_cat_mean_34.0', 'week_roll_weekly_change_cat_mean_11.0', 'week_roll_rolling_mean_cat_mean_35.0', 'week_roll_weekly_change_cat_mean_0.0', 'week_roll_weekly_change_cat_mean_1.0', 'week_roll_weekly_change_cat_mean_2.0', 'week_roll_weekly_change_cat_mean_3.0', 'week_roll_weekly_change_cat_mean_4.0', 'week_roll_weekly_change_cat_mean_5.0', 'week_roll_weekly_change_cat_mean_6.0', 'week_roll_weekly_change_cat_mean_7.0', 'week_roll_weekl